### LIBRARY IMPORTS

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.nn_regressor import NNRegressor

### CONFIGURATION

In [2]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_processed_data()

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

y_train, y_valid = processor.transform_target(y_train, y_valid)

### RIDGE & LASSO REGRESSION

In [7]:
ridge = RidgeCV(alphas=[0.1, 1, 10], cv=5)
ridge.fit(X_train, y_train)

ridge_test_preds = ridge.predict(X_valid)

print(f"Ridge best hyperparameter: {ridge.alpha_}")
print(f"Ridge validation MSE: {mean_squared_error(y_valid, ridge_test_preds)}")
print(f"Ridge validation R^2: {r2_score(y_valid, ridge_test_preds)}")

print('-' * 50)

lasso = LassoCV(alphas=[0.1, 1, 10], cv=5)
lasso.fit(X_train, y_train)

lasso_test_preds = lasso.predict(X_valid)

print(f"Lasso best hyperparameter: {lasso.alpha_}")
print(f"Lasso validation MSE: {mean_squared_error(y_valid, lasso_test_preds)}")
print(f"Lasso validation R^2: {r2_score(y_valid, lasso_test_preds)}")

Ridge best hyperparameter: 1.0
Ridge validation MSE: 0.10538682949285362
Ridge validation R^2: 0.7801616963641764
--------------------------------------------------
Lasso best hyperparameter: 0.1
Lasso validation MSE: 0.2146866172277347
Lasso validation R^2: 0.55216091069654


### RANDOM FOREST

In [9]:
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

rf_test_preds = rf.predict(X_valid)

print(f"Random forest validation MSE: {mean_squared_error(y_valid, rf_test_preds)}")
print(f"Random forest validation R^2: {r2_score(y_valid, rf_test_preds)}")

Random forest validation MSE: 0.1461784092172647
Random forest validation R^2: 0.6950699279487684


### NEURAL NETWORK

In [3]:
class MLP(NNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_valid: np.ndarray,
        y_valid: np.ndarray, 
        patience: int = 5
    ) -> None:
        
        input_size = X_train.shape[1]
        output_size = y_train.shape[1] if len(y_train.shape) > 1 else 1
        self._get_network(input_size, output_size)
        
        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).to(torch.float32).view(-1, output_size).to(self.device)

        criterion = nn.MSELoss()
        optimizer = optim.Adam(self.parameters(), self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                loss = criterion(preds, batch_y.view_as(preds))
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                val_preds_np = val_preds.cpu().numpy().flatten()
                y_valid_np = y_valid_t.cpu().numpy().flatten()
                val_r2 = r2_score(y_valid_np, val_preds_np)

            print(f"Epoch: {epoch+1} | Validation MSE: {val_loss:.4f} | R^2: {val_r2:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

In [9]:
mlp = MLP(epochs=100, learning_rate=0.0001, hidden_size=[64, 32], batch_size=256)
mlp.fit(X_train, y_train, X_valid, y_valid)

Epoch: 1 | Validation MSE: 0.1879 | R^2: 0.6080
Epoch: 2 | Validation MSE: 0.1105 | R^2: 0.7695
Epoch: 3 | Validation MSE: 0.1082 | R^2: 0.7742
Epoch: 4 | Validation MSE: 0.1073 | R^2: 0.7761
Epoch: 5 | Validation MSE: 0.1069 | R^2: 0.7770
Epoch: 6 | Validation MSE: 0.1065 | R^2: 0.7778
Epoch: 7 | Validation MSE: 0.1065 | R^2: 0.7778
Epoch: 8 | Validation MSE: 0.1062 | R^2: 0.7785
Epoch: 9 | Validation MSE: 0.1061 | R^2: 0.7786
Epoch: 10 | Validation MSE: 0.1060 | R^2: 0.7789
Epoch: 11 | Validation MSE: 0.1059 | R^2: 0.7790
Epoch: 12 | Validation MSE: 0.1059 | R^2: 0.7790
Epoch: 13 | Validation MSE: 0.1059 | R^2: 0.7791
Epoch: 14 | Validation MSE: 0.1058 | R^2: 0.7792
Epoch: 15 | Validation MSE: 0.1058 | R^2: 0.7793
Epoch: 16 | Validation MSE: 0.1060 | R^2: 0.7790
Epoch: 17 | Validation MSE: 0.1057 | R^2: 0.7795
Epoch: 18 | Validation MSE: 0.1057 | R^2: 0.7796
Epoch: 19 | Validation MSE: 0.1056 | R^2: 0.7797
Epoch: 20 | Validation MSE: 0.1056 | R^2: 0.7797
Epoch: 21 | Validation MSE: 0